# Silver Layer (Data Cleaning)
In this notebook, we will read the raw data from the `bronze` layer, clean it, and write it to the `silver` layer.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

print("Starting Spark Session...")
spark = SparkSession.builder \
    .appName("DataCore_Silver") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,io.unitycatalog:unitycatalog-spark_2.12:0.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity.uri", "http://unitycatalog:8080") \
    .config("spark.sql.defaultCatalog", "unity") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Session Ready!")

Starting Spark Session...
Spark Session Ready!


In [2]:
# 1. Read the raw data from Bronze
print("Reading from Bronze...")
df_bronze = spark.read.format("delta").load("s3://bronze/test_delta_table")
df_bronze.show()

# 2. Clean and Transform the Data (The Silver Layer)
# Example: We will filter out anyone under 24 years old, and add a timestamp column
df_silver = df_bronze.filter(col("Age") >= 24).withColumn("ingestion_time", current_timestamp())

print("Transformed Silver Data:")
df_silver.show()

Reading from Bronze...
+-----+---+
| Name|Age|
+-----+---+
|Ahmed| 25|
| Sara| 22|
| Omar| 30|
+-----+---+

Transformed Silver Data:
+-----+---+--------------------+
| Name|Age|      ingestion_time|
+-----+---+--------------------+
|Ahmed| 25|2026-09-29 12:24:...|
| Omar| 30|2026-09-29 12:24:...|
+-----+---+--------------------+



In [3]:
# 3. Write to MinIO directly (bypassing Unity Catalog credentials for now)
print("Writing to Silver Bucket...")
s3_silver_path = "s3://silver/users_clean"

df_silver.write.format("delta") \
    .mode("overwrite") \
    .save(s3_silver_path)

print("Success! Data written to Silver layer.")

Writing to Silver Bucket...
Success! Data written to Silver layer.


In [4]:
# 4. Verify that we can read the data back!
spark.read.format("delta").load(s3_silver_path).show()

+-----+---+--------------------+
| Name|Age|      ingestion_time|
+-----+---+--------------------+
|Ahmed| 25|2026-09-29 12:24:...|
| Omar| 30|2026-09-29 12:24:...|
+-----+---+--------------------+

